In [0]:
# ============================================================
# SCD TYPE 2 - CUSTOMER DIMENSION
# ============================================================

from pyspark.sql import functions as F
from delta.tables import DeltaTable

SOURCE_TABLE = "telecom.silver.customers"
TARGET_TABLE = "telecom.gold.dim_customer_scd2"

source_df = spark.table(SOURCE_TABLE)

# Prepare source data
scd_source = (
    source_df
    .select(
        "customer_id",
        "first_name",
        "last_name",
        "date_of_birth",
        "gender",
        "email",
        "registration_date",
        "customer_status",
        "created_at",
        "updated_at"
    )
    .withColumn(
        "effective_start_date",
        F.current_date()
    )
    .withColumn(
        "effective_end_date",
        F.lit(None).cast("date")
    )
    .withColumn(
        "is_current",
        F.lit(True)
    )
)

# ============================================================
# INITIAL LOAD
# ============================================================

if not spark.catalog.tableExists(TARGET_TABLE):

    (
        scd_source
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(TARGET_TABLE)
    )

    print("SCD Type 2 dimension created successfully")
    print(f"Records loaded: {scd_source.count()}")

else:

    target = DeltaTable.forName(
        spark,
        TARGET_TABLE
    )

    # --------------------------------------------------------
    # STEP 1: Detect changed current records
    # --------------------------------------------------------

    current_target = (
        spark.table(TARGET_TABLE)
        .filter(F.col("is_current") == True)
        .select(
            "customer_id",
            "first_name",
            "last_name",
            "date_of_birth",
            "gender",
            "email",
            "registration_date",
            "customer_status"
        )
    )

    changed_records = (
        scd_source.alias("s")
        .join(
            current_target.alias("t"),
            "customer_id",
            "inner"
        )
        .filter(
            (F.coalesce(F.col("s.first_name"), F.lit("")) !=
             F.coalesce(F.col("t.first_name"), F.lit("")))
            |
            (F.coalesce(F.col("s.last_name"), F.lit("")) !=
             F.coalesce(F.col("t.last_name"), F.lit("")))
            |
            (F.coalesce(F.col("s.email"), F.lit("")) !=
             F.coalesce(F.col("t.email"), F.lit("")))
            |
            (F.coalesce(F.col("s.gender"), F.lit("")) !=
             F.coalesce(F.col("t.gender"), F.lit("")))
            |
            (F.coalesce(F.col("s.customer_status"), F.lit("")) !=
             F.coalesce(F.col("t.customer_status"), F.lit("")))
        )
        .select("s.*")
    )

    changed_count = changed_records.count()

    print(f"Changed customer records detected: {changed_count}")

    # --------------------------------------------------------
    # STEP 2: Close old versions
    # --------------------------------------------------------

    if changed_count > 0:

        changed_ids = changed_records.select(
            "customer_id"
        ).distinct()

        (
            target.alias("t")
            .merge(
                changed_ids.alias("s"),
                """
                t.customer_id = s.customer_id
                AND t.is_current = true
                """
            )
            .whenMatchedUpdate(
                set={
                    "effective_end_date": "current_date()",
                    "is_current": "false"
                }
            )
            .execute()
        )

        # ----------------------------------------------------
        # STEP 3: Insert new versions
        # ----------------------------------------------------

        (
            changed_records
            .write
            .format("delta")
            .mode("append")
            .saveAsTable(TARGET_TABLE)
        )

        print("New SCD Type 2 versions inserted")

    # --------------------------------------------------------
    # STEP 4: Insert completely new customers
    # --------------------------------------------------------

    existing_ids = (
        spark.table(TARGET_TABLE)
        .select("customer_id")
        .distinct()
    )

    new_records = (
        scd_source
        .join(
            existing_ids,
            "customer_id",
            "left_anti"
        )
    )

    new_count = new_records.count()

    if new_count > 0:

        (
            new_records
            .write
            .format("delta")
            .mode("append")
            .saveAsTable(TARGET_TABLE)
        )

        print(f"New customers inserted: {new_count}")

    print("SCD Type 2 incremental processing completed")

In [0]:
# ============================================================
# SCD TYPE 2 VALIDATION
# ============================================================

scd_validation = spark.sql("""
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT customer_id) AS unique_customers,
    SUM(
        CASE
            WHEN is_current = true THEN 1
            ELSE 0
        END
    ) AS current_records,
    SUM(
        CASE
            WHEN is_current = false THEN 1
            ELSE 0
        END
    ) AS historical_records,
    SUM(
        CASE
            WHEN effective_end_date IS NOT NULL
             AND effective_end_date < effective_start_date
            THEN 1
            ELSE 0
        END
    ) AS invalid_date_ranges
FROM telecom.gold.dim_customer_scd2
""")

display(scd_validation)

In [0]:
# ============================================================
# INCREMENTAL PROCESSING
# ============================================================

from pyspark.sql import functions as F

SOURCE_TABLE = "telecom.silver.call_records"
TARGET_TABLE = "telecom.gold.fact_call_usage_incremental"

# Read Silver source
source_df = spark.table(SOURCE_TABLE)

print(f"Source records: {source_df.count()}")

# Check whether incremental target exists
target_exists = spark.catalog.tableExists(TARGET_TABLE)

if not target_exists:

    # Initial load
    (
        source_df
        .withColumn(
            "processed_at",
            F.current_timestamp()
        )
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(TARGET_TABLE)
    )

    print("Initial load completed")
    print(f"Records loaded: {source_df.count()}")

else:

    # Get already processed IDs
    processed_ids = (
        spark.table(TARGET_TABLE)
        .select("call_id")
        .distinct()
    )

    # Identify only new records
    incremental_df = (
        source_df
        .join(
            processed_ids,
            on="call_id",
            how="left_anti"
        )
        .withColumn(
            "processed_at",
            F.current_timestamp()
        )
    )

    new_count = incremental_df.count()

    print(f"New records detected: {new_count}")

    if new_count > 0:

        (
            incremental_df
            .write
            .format("delta")
            .mode("append")
            .saveAsTable(TARGET_TABLE)
        )

        print(f"Incremental records appended: {new_count}")

    else:

        print("No new records to process")

print("Incremental processing completed")

In [0]:
# ============================================================
# INCREMENTAL PROCESSING VALIDATION
# ============================================================

validation_df = spark.sql("""
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT call_id) AS distinct_call_ids,
    MIN(processed_at) AS first_processed_at,
    MAX(processed_at) AS last_processed_at
FROM telecom.gold.fact_call_usage_incremental
""")

display(validation_df)

In [0]:
# ============================================================
# INCREMENTAL PROCESSING AUDIT
# ============================================================

source_count = spark.table(
    "telecom.silver.call_records"
).count()

target_count = spark.table(
    "telecom.gold.fact_call_usage_incremental"
).count()

duplicate_count = (
    spark.table("telecom.gold.fact_call_usage_incremental")
    .groupBy("call_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

status = (
    "PASS"
    if source_count == target_count and duplicate_count == 0
    else "CHECK"
)

print("=" * 70)
print("INCREMENTAL PROCESSING VALIDATION")
print("=" * 70)
print(f"Source records       : {source_count}")
print(f"Target records       : {target_count}")
print(f"Duplicate call IDs   : {duplicate_count}")
print(f"Status               : {status}")
print("=" * 70)

In [0]:
# ============================================================
# SOURCE-TO-TARGET RECONCILIATION
# ============================================================

from pyspark.sql import functions as F

tables_to_check = [
    ("customers",
     "telecom.bronze.customers",
     "telecom.silver.customers"),

    ("customer_addresses",
     "telecom.bronze.customer_addresses",
     "telecom.silver.customer_addresses"),

    ("customer_contacts",
     "telecom.bronze.customer_contacts",
     "telecom.silver.customer_contacts"),

    ("mobile_plans",
     "telecom.bronze.mobile_plans",
     "telecom.silver.mobile_plans"),

    ("service_types",
     "telecom.bronze.service_types",
     "telecom.silver.service_types"),

    ("plan_services",
     "telecom.bronze.plan_services",
     "telecom.silver.plan_services"),

    ("subscriptions",
     "telecom.bronze.subscriptions",
     "telecom.silver.subscriptions"),

    ("subscription_services",
     "telecom.bronze.subscription_services",
     "telecom.silver.subscription_services"),

    ("call_records",
     "telecom.bronze.call_records",
     "telecom.silver.call_records"),

    ("sms_records",
     "telecom.bronze.sms_records",
     "telecom.silver.sms_records"),

    ("data_usage",
     "telecom.bronze.data_usage",
     "telecom.silver.data_usage"),

    ("bills",
     "telecom.bronze.bills",
     "telecom.silver.bills"),

    ("bill_items",
     "telecom.bronze.bill_items",
     "telecom.silver.bill_items"),

    ("payments",
     "telecom.bronze.payments",
     "telecom.silver.payments"),

    ("complaint_categories",
     "telecom.bronze.complaint_categories",
     "telecom.silver.complaint_categories"),

    ("complaints",
     "telecom.bronze.complaints",
     "telecom.silver.complaints"),

    ("service_areas",
     "telecom.bronze.service_areas",
     "telecom.silver.service_areas")
]

results = []

for table_name, source_table, target_table in tables_to_check:

    source_count = spark.table(source_table).count()
    target_count = spark.table(target_table).count()

    difference = source_count - target_count

    status = "PASS" if difference == 0 else "CHECK"

    results.append(
        (
            table_name,
            source_count,
            target_count,
            difference,
            status
        )
    )

reconciliation_df = spark.createDataFrame(
    results,
    [
        "table_name",
        "source_records",
        "target_records",
        "difference",
        "status"
    ]
)

display(
    reconciliation_df.orderBy("table_name")
)

In [0]:
# ============================================================
# RECONCILIATION SUMMARY
# ============================================================

summary = reconciliation_df.agg(
    F.count("*").alias("tables_checked"),
    F.sum(
        F.when(F.col("status") == "PASS", 1).otherwise(0)
    ).alias("passed_tables"),
    F.sum(
        F.when(F.col("status") != "PASS", 1).otherwise(0)
    ).alias("failed_tables")
)

display(summary)